# Combine the estimates generated in 3.1 to 3.6

## 0. Load packages

In [12]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
import xlwings as xw
import time, shutil, os
from typing import Tuple

In [13]:
base = Path("../output/2025/tables/Final_Full_CBCR_Datasets")
out_xlsx = base / "SOTJ_Combined_Aggregates_and_Country_byYear.xlsx"
out_xlsx = Path("C:/Users/aliso/Tax Justice Network Ltd/TJN - Shared Documents/Research team/Projects long-term/SOTJ/SOTJ_2025/SOTJ_Combined_Aggregates_and_Country_byYear.xlsx")
csv_out_dir = base 
csv_out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---------- helpers ----------
def read_agg(year: int, suffix: str, label: str) -> pd.DataFrame:
    """Read aggregate results for a year and tag with MNSs label (exact column name)."""
    fp = base / f"SOTJ_sample_aggregate_results_{year}{suffix}.csv"
    if not fp.exists():
        return pd.DataFrame()
    df = pd.read_csv(fp)
    df["MNSs"] = label
    if "year" not in df.columns:
        df["year"] = year
    return df

def build_agg_panels():
    """Aggregates across years for All and US MNEs (2016–2021)."""
    all_parts, us_parts = [], []
    for yr in range(2016, 2022):
        a = read_agg(yr, "", "All MNEs")
        u = read_agg(yr, "_USMNEs", "US MNEs")
        if not a.empty: all_parts.append(a)
        if not u.empty: us_parts.append(u)
    all_df = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()
    us_df  = pd.concat(us_parts,  ignore_index=True) if us_parts  else pd.DataFrame()
    if not all_df.empty: all_df = all_df.sort_values(["year", "MNSs"])
    if not us_df.empty:  us_df  = us_df.sort_values(["year", "MNSs"])
    return all_df, us_df

def load_and_aggregate_country(fp: Path, year: int) -> pd.DataFrame:
    """Country-level aggregation per year × iso_partner (sum numeric columns)."""
    if not fp.exists():
        return pd.DataFrame()
    df = pd.read_csv(fp)
    if "iso_partner" not in df.columns:
        raise ValueError(f"'iso_partner' column not found in {fp}")
    if "year" not in df.columns:
        df["year"] = year
    num_cols   = df.select_dtypes(include="number").columns.tolist()
    group_keys = ["year", "iso_partner"]
    agg_cols   = [c for c in num_cols if c not in group_keys]
    out = (df.groupby(group_keys, as_index=False)[agg_cols].sum()
           if agg_cols else df[group_keys].drop_duplicates())
    non_num = [c for c in out.columns if c not in agg_cols]
    return out[non_num + agg_cols].sort_values(["year", "iso_partner"])

def get_or_add_sheet(book: xw.Book, sheet_name: str) -> xw.main.Sheet:
    """Return a sheet by name if it exists; otherwise add it safely."""
    names = [sht.name for sht in book.sheets]
    if sheet_name in names:
        return book.sheets[names.index(sheet_name)]
    # Add after the last sheet (or just add if blank book)
    return book.sheets.add(name=sheet_name, after=book.sheets[-1] if len(book.sheets) else None)

def write_df_to_sheet(book: xw.Book, sheet_name: str, df: pd.DataFrame, start_cell: str = "A1"):
    """Create/locate sheet, clear only values (keep formatting), and write df."""
    if df is None or df.empty:
        return
    sht = get_or_add_sheet(book, sheet_name)
    # Clear only existing values in the used range (keeps formatting)
    try:
        if sht.used_range is not None:
            sht.used_range.clear_contents()
    except Exception:
        # Some Excel versions return a COM error if there's no proper used range yet
        pass
    # Write DataFrame starting at start_cell (headers on, index off)
    sht.range(start_cell).options(index=False, header=True).value = df



In [15]:
# ---------- build data ----------
agg_all, agg_us = build_agg_panels()

if not agg_all.empty:
    agg_all.to_csv(csv_out_dir / "SOTJ_aggregates_2016_2021_All_MNEs.csv", index=False)

if not agg_us.empty:
    agg_us.to_csv(csv_out_dir / "SOTJ_aggregates_2016_2021_US_MNEs.csv", index=False)

# ---------- open/create workbook and write ----------
app = xw.App(visible=False, add_book=False)
# Optional: make it a bit faster and avoid alerts
app.display_alerts = False
app.screen_updating = False

book = None
try:
    if out_xlsx.exists():
        book = app.books.open(str(out_xlsx))
    else:
        book = app.books.add()
        # Ensure at least one sheet exists (xlwings usually creates one automatically)
        if len(book.sheets) == 0:
            book.sheets.add("Sheet1")

    # Aggregates (stacked across years)
    write_df_to_sheet(book, "Aggregates_All_MNEs", agg_all)
    write_df_to_sheet(book, "Aggregates_US_MNEs",  agg_us)

    # Per-year country sheets
    for year in range(2016, 2022):
        # All MNEs per-country
        df_all = load_and_aggregate_country(base / f"SOTJ_sample_countries_{year}.csv", year)
        write_df_to_sheet(book, f"{year}_All_MNEs", df_all)
        # US MNEs per-country
        df_us  = load_and_aggregate_country(base / f"SOTJ_sample_countries_{year}_USMNEs.csv", year)
        write_df_to_sheet(book, f"{year}_US_MNEs", df_us)

    # Save updates without overwriting workbook formatting/theme/etc.
    book.save(str(out_xlsx))
finally:
    # Close and quit safely
    try:
        if book is not None:
            book.close()
    finally:
        try:
            app.quit()
        except Exception:
            pass

com_error: (-2147352567, 'Exception occurred.', (0, 'Microsoft Excel', "Cannot access 'SOTJ_Combined_Aggregates_and_Country_byYear.xlsx'.", 'xlmain11.chm', 0, -2146827284), None)

### Export as one big CSV

In [16]:
base = Path(r"../output/2025/tables/Final_Full_CBCR_Datasets")

years = range(2016, 2022)
parts = []

for y in years:
    fp = base / f"SOTJ_sample_countries_{y}.csv"
    if not fp.exists():
        print(f"Warning: missing file {fp}")
        continue
    df = pd.read_csv(fp)
    df["year"] = y
    parts.append(df)

if not parts:
    raise FileNotFoundError("No input files found for 2016–2021.")

out = pd.concat(parts, ignore_index=True)

out = out[["iso_partner", "partner_jurisdiction", "year",
    "negative_misalignment", "positive_misalignment",
    "theoretical_profit", "reported_profit",
    "tax_revenue_loss", "tax_revenue_gain",
    "tax_revenue_loss_pct_of_gvt_health_expenditure",
    "tax_revenue_loss_pct_of_total_tax_revenues",
    "tax_revenue_loss_caused_pct_of_total", "tax_revenue_loss_caused_usd",
    "tax_revenue_loss_suffered_pct_of_total",
    "etr_average_corrected", "cit", "tax_revenue_current_usd",
    "gvt_health_expenditure", "region_tjn", "ukt", "oecd", "oecd_oct",
    "nld_oct"]]

url = "https://api.worldbank.org/v2/country?format=json&per_page=400"
data = requests.get(url).json()[1]  # [0] has paging metadata

wb = pd.DataFrame([{
    "iso_partner": c["id"],                 # ISO3
    "income_level_id": c["incomeLevel"]["id"],   # LIC/LMC/UMC/HIC
    "income_group": c["incomeLevel"]["value"],   # Low income / ...
} for c in data])

out = out.merge(wb[["iso_partner", "income_group", "income_level_id"]], on="iso_partner", how="left")

    
# save
out_path = base / "SOTJ_sample_countries_2016_2021_stacked.csv"
out.to_csv(out_path, index=False)
print(f"Wrote: {out_path}")

Wrote: ..\output\2025\tables\Final_Full_CBCR_Datasets\SOTJ_sample_countries_2016_2021_stacked.csv


In [17]:
# CPI index (2021 = 100), as provided by the IMF for the US (all items) here: https://data.imf.org/en/Data-Explorer?datasetUrn=IMF.STA:CPI(5.0.0)
cpi_index = pd.Series({
    2015: 108.695722,
    2016: 110.0670089,
    2017: 112.4115573,
    2018: 115.1573032,
    2019: 117.2441955,
    2020: 118.6905016,
    2021: 124.2664138,
    2022: 134.2112062,
    2023: 139.7357936,
    2024: 143.857336
}, name="cpi").rename_axis("year").reset_index()

exchange_rates = pd.read_csv('../data/2025/raw/API_PA.NUS.FCRF_DS2_en_csv_v2_114.csv', skiprows=4)
				
def deflate_to_base(df: pd.DataFrame,
                    cpi_df: pd.DataFrame,
                    year_col: str = "year",
                    base_year: int = 2021,
                    extra_exclude: list[str] | None = None) -> pd.DataFrame:
    """
    Convert nominal values in year t to base_year USD using CPI:
        real_base = nominal_t * (CPI_base / CPI_t)

    Keeps `year_col` in the output and never deflates it.
    """
    if extra_exclude is None:
        extra_exclude = []

    # Preserve original order and a pristine copy of the year column
    original_cols = df.columns.tolist()
    orig_year = df[year_col].copy()

    df = df.copy()
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce")

    cpi_df = cpi_df.copy()
    cpi_df["year"] = pd.to_numeric(cpi_df["year"], errors="coerce")

    # CPI at base year
    try:
        cpi_base = float(cpi_df.loc[cpi_df["year"] == base_year, "cpi"].iloc[0])
    except IndexError:
        raise ValueError(f"CPI index missing for base year {base_year}.")

    merged = df.merge(cpi_df, left_on=year_col, right_on="year", how="left", suffixes=("", "_cpi"))

    if merged["cpi"].isna().any():
        missing_years = merged.loc[merged["cpi"].isna(), year_col].dropna().unique().tolist()
        raise ValueError(f"CPI index missing for years in data: {missing_years}")

    # Deflator: nominal_t -> real_base
    merged["_deflator"] = cpi_base / merged["cpi"]

    # Identify numeric columns to deflate, excluding identifiers and rates
    exclude_cols = set([year_col, "year", "cpi", "_deflator"] + extra_exclude)
    numeric_cols = merged.select_dtypes(include=[np.number]).columns
    target_cols = [c for c in numeric_cols if c not in exclude_cols]

    # Apply deflation
    merged[target_cols] = merged[target_cols].multiply(merged["_deflator"], axis=0)

    # Clean up helper columns and restore untouched year
    merged = merged.drop(columns=["year", "cpi", "_deflator"])
    merged[year_col] = orig_year  # guarantee year is unchanged

    # Return with original column order (plus any new ones appended at end)
    ordered = [c for c in original_cols if c in merged.columns] + [c for c in merged.columns if c not in original_cols]
    return merged[ordered]


In [18]:
# Bring everything to 2021 USD
out_deflated  = deflate_to_base(
    out,  cpi_index, year_col="year", base_year=2021,
    extra_exclude=['iso_partner', 'year','partner_jurisdiction',
       'etr_average_corrected', 'cit','region_tjn', 'ukt', 'oecd', 'oecd_oct',
       'nld_oct', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
       'tax_revenue_loss_pct_of_total_tax_revenues',
       'tax_revenue_loss_caused_pct_of_total',
       'tax_revenue_loss_suffered_pct_of_total','income_group', 'income_level_id']
)
# add exchange rates
fx2021 = (
    exchange_rates.loc[:, ["Country Code", "Country Name", "2021"]]
    .rename(columns={
        "Country Code": "iso_partner",             # must match your key
        "Country Name": "wb_country_name",
        "2021": "exchange_rate_2021"             # LCU per 1 USD (period average)
    })
)

# ensure numeric (some WDI CSVs load years as strings)
fx2021["exchange_rate_2021"] = pd.to_numeric(fx2021["exchange_rate_2021"], errors="coerce")

# --- Merge onto your per-country table and compute local amounts ------------
out_deflated_exchr = out_deflated.merge(fx2021, on="iso_partner", how="left")

In [19]:
# save
out_path_deflated = base / "SOTJ_sample_countries_2016_2021_stacked_2021USD.csv"
out_deflated_exchr.to_csv(out_path_deflated, index=False)
print(f"Wrote: {out_path_deflated}")

Wrote: ..\output\2025\tables\Final_Full_CBCR_Datasets\SOTJ_sample_countries_2016_2021_stacked_2021USD.csv


In [20]:
# ensure numeric (in case the columns are string-typed)
for col in ["tax_revenue_loss_caused_usd", "positive_misalignment"]:
    out_deflated[col] = pd.to_numeric(out_deflated[col], errors="coerce")

# aggregate over all years per iso_partner
agg = (
    out_deflated
    .groupby("iso_partner", as_index=False)
    .agg(
        tax_revenue_loss_caused_usd_total=("tax_revenue_loss_caused_usd", "sum"),
        positive_misalignment_total=("positive_misalignment", "sum"),
        illegitimate_gains_total=("tax_revenue_gain", "sum"),
    )
    .sort_values("illegitimate_gains_total", ascending=False)
    .reset_index(drop=True)
)

# peek
print(agg.head(10))


  iso_partner  tax_revenue_loss_caused_usd_total  positive_misalignment_total  \
0         CHN                      118794.129412                452843.312215   
1         SAU                       39878.961798                155541.114612   
2         HKG                      179193.163352                689018.504178   
3         GBR                      139670.940626                520387.580898   
4         CHE                      140936.664259                542545.745985   
5         IRL                       95832.526710                372848.504164   
6         NLD                      134425.971185                500322.222767   
7         SGP                      122952.479413                469736.309663   
8         CAN                       81285.096347                313328.458803   
9         JPN                       23465.331403                 86516.365567   

   illegitimate_gains_total  
0              78902.998312  
1              64530.604847  
2              484

### Bilateral export for IFF portal (assuming harm is distributed equally)

In [ ]:
# ---------- NEW: bilateral CSV export ----------
from pathlib import Path

def bilateral_loss_long(df_year: pd.DataFrame) -> pd.DataFrame:
    """
    Build long bilateral matrix for a single year from a per-country sheet.
    Requires columns: year, iso_partner, tax_revenue_loss, tax_revenue_loss_caused_pct_of_total
    Returns: year, iso_partner1_responsible, iso_partner2_suffering, amount_usd
    """
    need = {"year", "iso_partner", "profit", "tax_revenue_loss_caused_pct_of_total"}
    missing = need - set(df_year.columns)
    if missing:
        raise ValueError(f"Missing columns for bilateral calc: {missing}")

    yr = df_year["year"].iloc[0]

    shares = df_year[["iso_partner", "tax_revenue_loss_caused_pct_of_total"]].copy()
    # If provided as percent (e.g. up to 100), convert to fraction
    if shares["tax_revenue_loss_caused_pct_of_total"].max() > 1.0:
        shares["tax_revenue_loss_caused_pct_of_total"] /= 100.0
    shares.rename(columns={
        "iso_partner": "iso_responsible",
        "tax_revenue_loss_caused_pct_of_total": "share"
    }, inplace=True)

    losses = df_year[["iso_partner", "tax_revenue_loss"]].copy()
    losses.rename(columns={
        "iso_partner": "iso_affected",
        "tax_revenue_loss": "loss_musd"
    }, inplace=True)

    shares["key"] = 1
    losses["key"] = 1
    out = shares.merge(losses, on="key").drop(columns="key")
    out["taxloss_musd"] = out["share"] * out["loss_musd"]
    out.insert(0, "year", yr)

    return out[["year", "iso_responsible", "iso_affected", "taxloss_musd"]]\
             .sort_values(["year", "iso_responsible", "iso_affected"])

# Where to save the CSVs (change if you prefer another base)
loss_dir = (csv_out_dir / "loss_bilateral")  # or: base / "loss_bilateral"
loss_dir.mkdir(parents=True, exist_ok=True)

bilateral_all_parts = []
bilateral_us_parts  = []

for year in range(2016, 2022):
    # Load per-country aggregates (you already have these helpers)
    df_all = load_and_aggregate_country(base / f"SOTJ_sample_countries_{year}.csv", year)
    if not df_all.empty:
        try:
            bl_all = bilateral_loss_long(df_all)
            bilateral_all_parts.append(bl_all)
            bl_all.to_csv(loss_dir / f"loss_bilateral_{year}_All_MNEs.csv", index=False)
        except Exception as e:
            print(f"[Bilateral All MNEs {year}] skipped: {e}")

    df_us = load_and_aggregate_country(base / f"SOTJ_sample_countries_{year}_USMNEs.csv", year)
    if not df_us.empty:
        try:
            bl_us = bilateral_loss_long(df_us)
            bilateral_us_parts.append(bl_us)
            bl_us.to_csv(loss_dir / f"loss_bilateral_{year}_US_MNEs.csv", index=False)
        except Exception as e:
            print(f"[Bilateral US MNEs {year}] skipped: {e}")

# Stacked (all years)
if bilateral_all_parts:
    pd.concat(bilateral_all_parts, ignore_index=True)\
      .to_csv(loss_dir / "loss_bilateral_All_MNEs_all_years.csv", index=False)

if bilateral_us_parts:
    pd.concat(bilateral_us_parts, ignore_index=True)\
      .to_csv(loss_dir / "loss_bilateral_US_MNEs_all_years.csv", index=False)
